
## SECTION 1: IMPORT LIBRARIES


In [1]:
import os
import random
import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import defaultdict
from tabulate import tabulate
import cv2

import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.cuda import amp
import torchvision
from torchvision.datasets import CocoDetection
from torchvision.transforms import functional as F
from torchvision.transforms import ColorJitter
from torchvision.models.detection import centernet_resnet50_fpn_v2
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as maskUtil

ImportError: cannot import name 'centernet_resnet50_fpn_v2' from 'torchvision.models.detection' (/home/somel/code/FYP_Project/.venv/lib/python3.12/site-packages/torchvision/models/detection/__init__.py)

## SECTION 2: CONFIGURATION


In [ ]:
# Dataset & Output Paths (adjust to your directory structure)
ROOT_DIR = "/home/somel/code/FYP_Project/Dataset/COCO/centernet_dataset.coco"
CHECKPOINT_DIR = "/home/somel/code/FYP_Project/Training_checkpoints/CenterNet/centernet_run1"
RESULTS_DIR = "/home/somel/code/FYP_Project/results/centernet_run1"

# Training hyperparameters
BATCH_SIZE = 2
ACCUMULATION_STEPS = 8  # Effective batch size = BATCH_SIZE * ACCUMULATION_STEPS
NUM_EPOCHS = 80
LEARNING_RATE = 0.001  # CenterNet typically benefits from slightly lower LR
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
NUM_WORKERS = 4

# Create directories
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Device setup
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
torch.cuda.empty_cache()

print(f"Using device: {device}")

## SECTION 3: DATASET PREPARATION

In [ ]:
def get_transform_function(train):
    """Apply augmentations for training, basic transform for validation."""
    color_jitter = ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1)

    def transform_func(image, target):
        image = F.to_tensor(image)
        if train:
            image = color_jitter(image)
            if random.random() > 0.5:
                image = F.hflip(image)
                width = image.shape[-1]
                for ann in target:
                    # Flip bounding box
                    x, y, w, h = ann['bbox']
                    ann['bbox'] = [width - x - w, y, w, h]
        return image, target
    return transform_func

def process_target(target, image_size=None, coco_id_mapping=None):
    """Convert COCO annotations to CenterNet format (boxes + labels only)."""
    boxes, labels = [], []
    
    for ann in target:
        x, y, w, h = ann['bbox']
        if w > 0 and h > 0:
            boxes.append([x, y, x + w, y + h])
            if coco_id_mapping is not None:
                labels.append(coco_id_mapping[ann['category_id']])
            else:
                labels.append(ann['category_id'])

    if not boxes:
        boxes = torch.zeros((0, 4), dtype=torch.float32)
        labels = torch.zeros((0,), dtype=torch.int64)
    else:
        boxes = torch.tensor(boxes, dtype=torch.float32).view(-1, 4)
        labels = torch.tensor(labels, dtype=torch.int64)

    return {'boxes': boxes, 'labels': labels}

## SECTION 4: DATA SPLITTING / LOADING

In [ ]:
print("Setting up Data Loaders...")
train_images_dir = os.path.join(ROOT_DIR, "train")
train_ann_file = os.path.join(ROOT_DIR, "train", "_annotations.coco.json")
val_images_dir = os.path.join(ROOT_DIR, "valid")
val_ann_file = os.path.join(ROOT_DIR, "valid", "_annotations.coco.json")

train_dataset = CocoDetection(root=train_images_dir, annFile=train_ann_file,
                              transforms=get_transform_function(train=True))
val_dataset = CocoDetection(root=val_images_dir, annFile=val_ann_file,
                            transforms=get_transform_function(train=False))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, collate_fn=lambda x: tuple(zip(*x)))

# Load COCO ground truth for evaluation
coco_gt = COCO(val_ann_file)

# Build dynamic mapping: COCO category_id -> model label (1-indexed)
coco_categories = coco_gt.loadCats(coco_gt.getCatIds())
sorted_categories = sorted(coco_categories, key=lambda x: x['id'])
COCO_ID_TO_MODEL_LABEL = {cat['id']: idx + 1 for idx, cat in enumerate(sorted_categories)}
MODEL_LABEL_TO_COCO_ID = {v: k for k, v in COCO_ID_TO_MODEL_LABEL.items()}
MODEL_LABEL_TO_NAME = {idx + 1: cat['name'] for idx, cat in enumerate(sorted_categories)}
CLASS_NAMES = [cat['name'] for cat in sorted_categories]

# Update NUM_CLASSES dynamically (background + actual classes)
NUM_CLASSES = len(CLASS_NAMES) + 1
print(f"COCO ID -> Model Label: {COCO_ID_TO_MODEL_LABEL}")
print(f"Classes: {CLASS_NAMES}")
print(f"NUM_CLASSES: {NUM_CLASSES}")

## SECTION 5: MODEL SETUP

In [ ]:
print(f"Loading CenterNet ResNet50-FPN v2 with {NUM_CLASSES} classes...")

# CenterNet in torchvision supports num_classes directly.
# Note: pretrained weights expect 80 classes. For custom classes, we initialize fresh or adapt carefully.
model = centernet_resnet50_fpn_v2(num_classes=NUM_CLASSES)
model.to(device)

# Optimizer with SGD
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

# Adaptive LR scheduler - reduces LR when validation mAP plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

# Mixed precision scaler
scaler = amp.GradScaler()

# Training history
history = {
    'train_loss': [],
    'loss_hm': [], 'loss_wh': [], 'loss_offset': [],  # CenterNet specific losses
    'precision': [], 'recall': [],
    'box_mAP50': [], 'box_mAP50-95': [],
    'lr': [], 'epoch_time': []
}


## SECTION 6: CHECKPOINT RESUME

In [ ]:
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "last_checkpoint.pth")
start_epoch = 0

if os.path.exists(CHECKPOINT_PATH):
    print(f"Loading checkpoint from {CHECKPOINT_PATH}...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    if 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    if 'history' in checkpoint:
        history = checkpoint['history']
    start_epoch = checkpoint['epoch'] + 1
    print(f"Resuming from Epoch {start_epoch}")

## SECTION 7: EVALUATION FUNCTIONS

In [ ]:
def collect_predictions(model, loader, device, conf_thresh=0.001):
    """Collect all predictions and ground truths for evaluation."""
    model.eval()
    all_preds = []
    all_gts = []

    with torch.no_grad():
        for batch_idx, (images, targets) in enumerate(loader):
            images = [img.to(device) for img in images]
            outputs = model(images)

            for i, output in enumerate(outputs):
                dataset_idx = batch_idx * loader.batch_size + i
                if dataset_idx >= len(loader.dataset):
                    continue
                image_id = loader.dataset.ids[dataset_idx]

                # Predictions
                boxes = output['boxes'].cpu().numpy()
                labels = output['labels'].cpu().numpy()
                scores = output['scores'].cpu().numpy()

                for j in range(len(boxes)):
                    if scores[j] >= conf_thresh:
                        all_preds.append({
                            'image_id': image_id, 'class_id': int(labels[j]),
                            'confidence': float(scores[j]), 'box': boxes[j].tolist()
                        })

                # Ground truths
                gt = process_target(targets[i], coco_id_mapping=COCO_ID_TO_MODEL_LABEL)
                gt_boxes = gt['boxes'].numpy()
                gt_labels = gt['labels'].numpy()
                for j in range(len(gt_boxes)):
                    all_gts.append({
                        'image_id': image_id, 'class_id': int(gt_labels[j]),
                        'box': gt_boxes[j].tolist()
                    })

    return all_preds, all_gts

def compute_iou(box1, box2):
    """Compute IoU between two boxes [x1, y1, x2, y2]."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter

    return inter / union if union > 0 else 0

def compute_metrics_at_threshold(preds, gts, conf_thresh, iou_thresh=0.5):
    """Compute precision, recall, F1 at a specific confidence threshold."""
    filtered_preds = [p for p in preds if p['confidence'] >= conf_thresh]

    gt_by_img_cls = defaultdict(list)
    for gt in gts:
        gt_by_img_cls[(gt['image_id'], gt['class_id'])].append(gt)

    tp, fp, total_gt = 0, 0, len(gts)
    matched_gts = set()

    filtered_preds = sorted(filtered_preds, key=lambda x: -x['confidence'])

    for pred in filtered_preds:
        key = (pred['image_id'], pred['class_id'])
        best_iou, best_gt_idx = 0, -1

        for idx, gt in enumerate(gt_by_img_cls[key]):
            iou = compute_iou(pred['box'], gt['box'])
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = idx

        gt_key = (key[0], key[1], best_gt_idx)
        if best_iou >= iou_thresh and gt_key not in matched_gts:
            tp += 1
            matched_gts.add(gt_key)
        else:
            fp += 1

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / total_gt if total_gt > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1

def compute_ap(preds, gts, class_id, iou_thresh=0.5):
    """Compute Average Precision for a single class."""
    class_preds = [p for p in preds if p['class_id'] == class_id]
    class_gts = [g for g in gts if g['class_id'] == class_id]

    if not class_gts:
        return 0.0, np.array([]), np.array([])

    class_preds = sorted(class_preds, key=lambda x: -x['confidence'])

    gt_by_img = defaultdict(list)
    for gt in class_gts:
        gt_by_img[gt['image_id']].append(gt)

    tp_list, fp_list = [], []
    matched = set()

    for pred in class_preds:
        img_gts = gt_by_img[pred['image_id']]
        best_iou, best_idx = 0, -1

        for idx, gt in enumerate(img_gts):
            iou = compute_iou(pred['box'], gt['box'])
            if iou > best_iou:
                best_iou = iou
                best_idx = idx

        key = (pred['image_id'], best_idx)
        if best_iou >= iou_thresh and key not in matched:
            tp_list.append(1)
            fp_list.append(0)
            matched.add(key)
        else:
            tp_list.append(0)
            fp_list.append(1)

    tp_cumsum = np.cumsum(tp_list)
    fp_cumsum = np.cumsum(fp_list)

    recalls = tp_cumsum / len(class_gts)
    precisions = tp_cumsum / (tp_cumsum + fp_cumsum)

    # Compute AP using 11-point interpolation
    ap = 0
    for t in np.linspace(0, 1, 11):
        prec_at_recall = precisions[recalls >= t]
        ap += prec_at_recall.max() if len(prec_at_recall) > 0 else 0
    ap /= 11

    return ap, recalls, precisions

def validate_epoch(model, loader, coco_gt, device):
    """Run COCO evaluation for boxes, return mAP values."""
    model.eval()
    box_predictions = []

    with torch.no_grad():
        for batch_idx, (images, targets) in enumerate(loader):
            images = [img.to(device) for img in images]
            outputs = model(images)

            for i, output in enumerate(outputs):
                dataset_idx = batch_idx * loader.batch_size + i
                if dataset_idx >= len(loader.dataset):
                    continue
                image_id = loader.dataset.ids[dataset_idx]

                boxes = output['boxes'].cpu().numpy()
                labels = output['labels'].cpu().numpy()
                scores = output['scores'].cpu().numpy()

                for j in range(len(boxes)):
                    x1, y1, x2, y2 = boxes[j]
                    coco_cat_id = MODEL_LABEL_TO_COCO_ID.get(int(labels[j]), int(labels[j]))
                    box_predictions.append({
                        'image_id': int(image_id), 'category_id': coco_cat_id,
                        'bbox': [float(x1), float(y1), float(x2-x1), float(y2-y1)],
                        'score': float(scores[j])
                    })

    # Default values
    box_mAP50, box_mAP50_95 = 0.0, 0.0
    precision, recall = 0.0, 0.0

    if box_predictions:
        coco_dt_box = coco_gt.loadRes(box_predictions)
        coco_eval_box = COCOeval(coco_gt, coco_dt_box, 'bbox')
        coco_eval_box.params.imgIds = loader.dataset.ids
        coco_eval_box.evaluate()
        coco_eval_box.accumulate()
        coco_eval_box.summarize()
        box_mAP50 = coco_eval_box.stats[1]
        box_mAP50_95 = coco_eval_box.stats[0]

    # Estimate precision/recall at IoU=0.5, conf=0.5
    preds, gts = collect_predictions(model, loader, device, conf_thresh=0.5)
    precision, recall, _ = compute_metrics_at_threshold(preds, gts, conf_thresh=0.5)

    return box_mAP50, box_mAP50_95, precision, recall

## SECTION 8: PLOTTING FUNCTIONS

In [ ]:
def save_training_dashboard(history, save_dir):
    """Generate training dashboard with 10 subplots."""
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    epochs = range(1, len(history['train_loss']) + 1)

    # Row 1: Training losses
    loss_keys = ['loss_hm', 'loss_wh', 'loss_offset', 'train_loss', 'lr']
    for i, key in enumerate(loss_keys):
        if key in history and len(history[key]) > 0:
            axes[0, i].plot(epochs, history[key], 'b.-')
            axes[0, i].set_title(key.replace('loss_', 'train/') if 'loss' in key else key)
            axes[0, i].set_xlabel('Epoch')

    # Row 2: Validation metrics
    val_keys = ['precision', 'recall', 'box_mAP50', 'box_mAP50-95']
    for i, key in enumerate(val_keys):
        if key in history and len(history[key]) > 0:
            axes[1, i].plot(epochs, history[key], 'b.-')
            axes[1, i].set_title(f'metrics/{key}')
            axes[1, i].set_xlabel('Epoch')

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'results.png'), dpi=150)
    plt.close()

def save_pr_curve(preds, gts, class_names, save_dir):
    """Save Precision-Recall curve per class."""
    plt.figure(figsize=(10, 8))
    class_ids = sorted(set(g['class_id'] for g in gts))
    all_aps = []

    for cls_id in class_ids:
        ap, recalls, precisions = compute_ap(preds, gts, cls_id)
        all_aps.append(ap)
        cls_name = MODEL_LABEL_TO_NAME.get(cls_id, f"Class {cls_id}")
        if len(recalls) > 0:
            plt.plot(recalls, precisions, label=f'{cls_name} {ap:.3f}', linewidth=1.5)

    mean_ap = np.mean(all_aps) if all_aps else 0
    plt.plot([0, 1], [mean_ap, mean_ap], 'b-', linewidth=2.5, label=f'all classes {mean_ap:.3f} mAP@0.5')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend(loc='upper right')
    plt.xlim([0, 1])
    plt.ylim([0, 1])
    plt.grid(True, alpha=0.3)
    plt.savefig(os.path.join(save_dir, 'PR_curve.png'), dpi=150)
    plt.close()

def save_confidence_curves(preds, gts, class_names, save_dir):
    """Save P-Conf, R-Conf, F1-Conf curves."""
    conf_thresholds = np.linspace(0, 1, 101)
    class_ids = sorted(set(g['class_id'] for g in gts))

    class_metrics = {cls_id: {'precision': [], 'recall': [], 'f1': []} for cls_id in class_ids}
    all_metrics = {'precision': [], 'recall': [], 'f1': []}

    for conf in conf_thresholds:
        p, r, f = compute_metrics_at_threshold(preds, gts, conf)
        all_metrics['precision'].append(p)
        all_metrics['recall'].append(r)
        all_metrics['f1'].append(f)

        for cls_id in class_ids:
            cls_preds = [p for p in preds if p['class_id'] == cls_id]
            cls_gts = [g for g in gts if g['class_id'] == cls_id]
            p, r, f = compute_metrics_at_threshold(cls_preds, cls_gts, conf)
            class_metrics[cls_id]['precision'].append(p)
            class_metrics[cls_id]['recall'].append(r)
            class_metrics[cls_id]['f1'].append(f)

    # Precision-Confidence
    plt.figure(figsize=(10, 8))
    for cls_id in class_ids:
        cls_name = MODEL_LABEL_TO_NAME.get(cls_id, f"Class {cls_id}")
        plt.plot(conf_thresholds, class_metrics[cls_id]['precision'], label=cls_name, linewidth=1.5)
    best_idx = np.argmax(all_metrics['precision'])
    plt.plot(conf_thresholds, all_metrics['precision'], 'b-', linewidth=2.5,
             label=f'all classes {all_metrics["precision"][best_idx]:.2f} at {conf_thresholds[best_idx]:.3f}')
    plt.xlabel('Confidence')
    plt.ylabel('Precision')
    plt.title('Precision-Confidence Curve')
    plt.legend(loc='lower left')
    plt.grid(True, alpha=0.3)
    plt.savefig(os.path.join(save_dir, 'P_curve.png'), dpi=150)
    plt.close()

    # Recall-Confidence
    plt.figure(figsize=(10, 8))
    for cls_id in class_ids:
        cls_name = MODEL_LABEL_TO_NAME.get(cls_id, f"Class {cls_id}")
        plt.plot(conf_thresholds, class_metrics[cls_id]['recall'], label=cls_name, linewidth=1.5)
    best_idx = np.argmax(all_metrics['recall'])
    plt.plot(conf_thresholds, all_metrics['recall'], 'b-', linewidth=2.5,
             label=f'all classes {all_metrics["recall"][best_idx]:.2f} at {conf_thresholds[best_idx]:.3f}')
    plt.xlabel('Confidence')
    plt.ylabel('Recall')
    plt.title('Recall-Confidence Curve')
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.savefig(os.path.join(save_dir, 'R_curve.png'), dpi=150)
    plt.close()

    # F1-Confidence
    plt.figure(figsize=(10, 8))
    for cls_id in class_ids:
        cls_name = MODEL_LABEL_TO_NAME.get(cls_id, f"Class {cls_id}")
        plt.plot(conf_thresholds, class_metrics[cls_id]['f1'], label=cls_name, linewidth=1.5)
    best_idx = np.argmax(all_metrics['f1'])
    plt.plot(conf_thresholds, all_metrics['f1'], 'b-', linewidth=2.5,
             label=f'all classes {all_metrics["f1"][best_idx]:.2f} at {conf_thresholds[best_idx]:.3f}')
    plt.xlabel('Confidence')
    plt.ylabel('F1')
    plt.title('F1-Confidence Curve')
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.savefig(os.path.join(save_dir, 'F1_curve.png'), dpi=150)
    plt.close()

def compute_confusion_matrix_no_bg(model, loader, device, num_classes, class_names, iou_threshold=0.5, conf_thresh=0.5):
    """Compute confusion matrix with background column/row for FP/FN."""
    model.eval()
    num_real = num_classes - 1
    cm = np.zeros((num_real + 1, num_real + 1), dtype=np.int64)
    bg_idx = num_real

    with torch.no_grad():
        for images, targets in loader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            for i, output in enumerate(outputs):
                gt = process_target(targets[i], coco_id_mapping=COCO_ID_TO_MODEL_LABEL)
                gt_boxes  = gt['boxes'].to(device)
                gt_labels = gt['labels'].to(device)

                pred_boxes  = output['boxes']
                pred_labels = output['labels']
                pred_scores = output['scores']

                keep = pred_scores > conf_thresh
                pred_boxes  = pred_boxes[keep]
                pred_labels = pred_labels[keep]

                matched_gt_indices   = set()
                matched_pred_indices = set()

                if len(gt_boxes) > 0 and len(pred_boxes) > 0:
                    ious = torchvision.ops.box_iou(gt_boxes, pred_boxes)

                    for g_idx in range(len(gt_boxes)):
                        iou_row = ious[g_idx].clone()
                        for used in matched_pred_indices:
                            iou_row[used] = 0.0
                        best_iou, best_p_idx = iou_row.max(dim=0)

                        if best_iou >= iou_threshold:
                            matched_gt_indices.add(g_idx)
                            matched_pred_indices.add(best_p_idx.item())
                            g_lab = gt_labels[g_idx].item() - 1
                            p_lab = pred_labels[best_p_idx].item() - 1
                            if 0 <= g_lab < num_real and 0 <= p_lab < num_real:
                                cm[g_lab, p_lab] += 1

                for g_idx in range(len(gt_boxes)):
                    if g_idx not in matched_gt_indices:
                        g_lab = gt_labels[g_idx].item() - 1
                        if 0 <= g_lab < num_real:
                            cm[g_lab, bg_idx] += 1

                for p_idx in range(len(pred_boxes)):
                    if p_idx not in matched_pred_indices:
                        p_lab = pred_labels[p_idx].item() - 1
                        if 0 <= p_lab < num_real:
                            cm[bg_idx, p_lab] += 1

    return cm

def save_confusion_matrix(cm, class_names, save_dir, include_background=True):
    """Save confusion matrix heatmap."""
    display_names = list(class_names) + ['Background (FP/FN)'] if include_background else list(class_names)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=display_names, yticklabels=display_names)
    plt.xlabel('Predicted')
    plt.ylabel('Ground Truth')
    plt.title('Confusion Matrix (Last row/col = unmatched FP/FN)')
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'confusion_matrix.png'), dpi=150)
    plt.close()

def save_metrics_table(history, preds, gts, class_names, save_dir, coco_box_mAP50=None, class_ids_to_show=None):
    """Save metrics summary table."""
    class_ids = sorted(set(g['class_id'] for g in gts))
    if class_ids_to_show is not None:
        class_ids = [c for c in class_ids if c in class_ids_to_show]

    table_data = []
    for cls_id in class_ids:
        cls_name = MODEL_LABEL_TO_NAME.get(cls_id, f"Class {cls_id}")
        ap, _, _ = compute_ap(preds, gts, cls_id)
        cls_preds = [p for p in preds if p['class_id'] == cls_id]
        cls_gts   = [g for g in gts   if g['class_id'] == cls_id]
        p, r, f1  = compute_metrics_at_threshold(cls_preds, cls_gts, 0.5)
        table_data.append([cls_name, f"{p:.4f}", f"{r:.4f}", f"{f1:.4f}", f"{ap:.4f}"])

    all_aps  = [compute_ap(preds, gts, c)[0] for c in class_ids]
    mean_ap  = np.mean(all_aps) if all_aps else 0.0
    filtered_preds = [p for p in preds if p['class_id'] in class_ids]
    filtered_gts   = [g for g in gts   if g['class_id'] in class_ids]
    overall_p, overall_r, overall_f1 = compute_metrics_at_threshold(filtered_preds, filtered_gts, 0.5)
    table_data.append(['Overall', f"{overall_p:.4f}", f"{overall_r:.4f}", f"{overall_f1:.4f}", f"{mean_ap:.4f}"])

    headers = ['Class', 'Precision', 'Recall', 'F1', 'mAP@0.5']
    print("\n" + "="*60)
    print("METRICS SUMMARY")
    print("="*60)
    print(tabulate(table_data, headers=headers, tablefmt='grid'))
    if coco_box_mAP50 is not None:
        print(f"\n[Reference] COCO box mAP@0.5 (pycocotools, last epoch): {coco_box_mAP50:.4f}")

    df = pd.DataFrame(table_data, columns=headers)
    df.to_csv(os.path.join(save_dir, 'metrics_summary.csv'), index=False)

    if history['train_loss']:
        print(f"\nFinal Training Loss:     {history['train_loss'][-1]:.4f}")
        print(f"Final Box mAP@0.5:       {history['box_mAP50'][-1]:.4f}")
        print(f"Final Box mAP@0.5:0.95:  {history['box_mAP50-95'][-1]:.4f}")

def save_results_csv(history, save_dir):
    """Save training history to CSV."""
    filtered_history = {k: v for k, v in history.items() if len(v) > 0}
    df = pd.DataFrame(filtered_history)
    df.index.name = 'epoch'
    df.to_csv(os.path.join(save_dir, 'results.csv'))

## SECTION 9: TRAINING LOOP

In [ ]:
print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60)
training_start_time = time.time()

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_start_time = time.time()
    model.train()
    epoch_loss = 0.0
    running_losses = defaultdict(float)

    optimizer.zero_grad()

    for batch_idx, (images, targets) in enumerate(train_loader):
        # Learning rate warmup for first epoch
        if epoch == 0 and batch_idx < 500:
            lr_factor = (batch_idx + 1) / 500
            for pg in optimizer.param_groups:
                pg['lr'] = lr_factor * LEARNING_RATE

        targets = [process_target(t, coco_id_mapping=COCO_ID_TO_MODEL_LABEL) for t in targets]
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        with amp.autocast():
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

        # Gradient accumulation
        loss_accum = losses / ACCUMULATION_STEPS
        scaler.scale(loss_accum).backward()

        if (batch_idx + 1) % ACCUMULATION_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        epoch_loss += losses.item()
        for k, v in loss_dict.items():
            running_losses[k] += v.item()

        if batch_idx % 20 == 0:
            print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Batch {batch_idx}, Loss: {losses.item():.4f}, LR: {optimizer.param_groups[0]['lr']:.2e}")

    avg_epoch_loss = epoch_loss / len(train_loader)
    current_lr = optimizer.param_groups[0]['lr']
    epoch_time = time.time() - epoch_start_time
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} Complete - Avg Loss: {avg_epoch_loss:.4f} - Time: {epoch_time:.1f}s")

    # Save training losses
    history['train_loss'].append(avg_epoch_loss)
    history['lr'].append(current_lr)
    history['epoch_time'].append(epoch_time)
    for k in running_losses:
        history[k].append(running_losses[k] / len(train_loader))

    # Validation
    print("Running validation...")
    box_mAP50, box_mAP50_95, precision, recall = validate_epoch(model, val_loader, coco_gt, device)
    history['box_mAP50'].append(box_mAP50)
    history['box_mAP50-95'].append(box_mAP50_95)
    history['precision'].append(precision)
    history['recall'].append(recall)

    print(f"Box mAP@0.5: {box_mAP50:.4f}, Box mAP@0.5:0.95: {box_mAP50_95:.4f}")
    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}")

    # Update scheduler
    scheduler.step(box_mAP50)

    # Save checkpoint
    checkpoint_data = {
        'epoch': epoch, 'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'history': history, 'loss': avg_epoch_loss
    }
    torch.save(checkpoint_data, CHECKPOINT_PATH)

    if box_mAP50 >= max(history['box_mAP50']):
        torch.save(checkpoint_data, os.path.join(CHECKPOINT_DIR, 'best.pth'))
        print("New best model saved!")

    save_training_dashboard(history, RESULTS_DIR)
    save_results_csv(history, RESULTS_DIR)
    print(f"Checkpoint saved for Epoch {epoch+1}\n")

total_training_time = time.time() - training_start_time
hours = int(total_training_time // 3600)
minutes = int((total_training_time % 3600) // 60)
seconds = int(total_training_time % 60)
print(f"Total Training Time: {hours}h {minutes}m {seconds}s")

## SECTION 10: FINAL EVALUATION

In [ ]:
print("\n" + "="*60)
print("TRAINING COMPLETE - FINAL EVALUATION")
print("="*60)

# 1. Collect predictions
print("Collecting predictions for evaluation curves...")
preds, gts = collect_predictions(model, val_loader, device)

# 2. Define which classes to keep
CLASSES_TO_KEEP  = [name for name in CLASS_NAMES]  # Keep all, or filter like ['Groupers', 'Seabass']
keep_indices     = [i for i, name in enumerate(CLASS_NAMES) if name in CLASSES_TO_KEEP]
filtered_class_names = [CLASS_NAMES[i] for i in keep_indices]
keep_class_ids = [cls_id for cls_id, name in MODEL_LABEL_TO_NAME.items() if name in CLASSES_TO_KEEP]
print(f"Keeping classes: {filtered_class_names}")
print(f"Corresponding class IDs: {keep_class_ids}")

# 3. PR curve
print("Generating Precision-Recall curve...")
save_pr_curve(preds, gts, CLASS_NAMES, RESULTS_DIR)

# 4. Confidence curves
print("Generating confidence curves...")
save_confidence_curves(preds, gts, CLASS_NAMES, RESULTS_DIR)

# 5. Confusion matrix
print("Computing confusion matrix...")
cm = compute_confusion_matrix_no_bg(model, val_loader, device, NUM_CLASSES, CLASS_NAMES)
bg_idx        = len(CLASS_NAMES)
row_col_indices = keep_indices + [bg_idx]
cm_filtered   = cm[np.ix_(row_col_indices, row_col_indices)]
save_confusion_matrix(cm_filtered, filtered_class_names, RESULTS_DIR, include_background=True)

# 6. Metrics table
print("Generating metrics summary table...")
save_metrics_table(
    history, preds, gts, CLASS_NAMES, RESULTS_DIR,
    coco_box_mAP50=history['box_mAP50'][-1],
    class_ids_to_show=keep_class_ids
)

# 7. Sanity check
print("\n" + "="*60)
print("SANITY CHECK — CM precision vs Table precision")
print("="*60)
for i, name in enumerate(filtered_class_names):
    col_sum      = cm_filtered[:, i].sum()
    cm_precision = cm_filtered[i, i] / col_sum if col_sum > 0 else 0.0
    cls_id    = keep_class_ids[i]
    cls_preds = [p for p in preds if p['class_id'] == cls_id]
    cls_gts   = [g for g in gts   if g['class_id'] == cls_id]
    tbl_p, tbl_r, _ = compute_metrics_at_threshold(cls_preds, cls_gts, 0.5)
    match = "✅" if abs(cm_precision - tbl_p) < 0.05 else "⚠️  MISMATCH"
    print(f"  {name:<12}  CM precision: {cm_precision:.4f}   Table precision: {tbl_p:.4f}   {match}")

print(f"\nAll results saved to: {RESULTS_DIR}")

## SECTION 11: IMAGE INFERENCE

In [ ]:
def inference_image(model, image_path, device, conf_threshold=0.5):
    """Run inference on a single image and display results with bounding boxes."""
    model.eval()
    image = Image.open(image_path).convert('RGB')
    image_np = np.array(image)
    image_tensor = F.to_tensor(image).to(device)

    with torch.no_grad():
        outputs = model([image_tensor])[0]

    keep = outputs['scores'] > conf_threshold
    boxes  = outputs['boxes'][keep].cpu().numpy()
    labels = outputs['labels'][keep].cpu().numpy()
    scores = outputs['scores'][keep].cpu().numpy()

    CLASS_COLORS_RGB = {1: (255, 200, 100), 2: (0, 255, 0), 3: (255, 0, 0), 4: (0, 100, 255)}
    CLASS_COLORS_HEX = {1: '#FFC864', 2: '#00FF00', 3: '#FF0000', 4: '#0064FF'}

    overlay = image_np.copy().astype(np.float32)
    # Note: CenterNet doesn't output masks, so we skip mask overlays.

    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(overlay.astype(np.uint8))

    for box, label, score in zip(boxes, labels, scores):
        x1, y1, x2, y2 = box
        color = CLASS_COLORS_HEX.get(label, '#C8C8C8')
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor=color, linewidth=2)
        ax.add_patch(rect)
        cls_name = MODEL_LABEL_TO_NAME.get(label, f"Class {label}")
        ax.text(x1, y1-5, f'{cls_name}: {score:.2f}', color=color, fontsize=10,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    ax.axis('off')
    plt.title(f'CenterNet Detection - {os.path.basename(image_path)}')
    plt.tight_layout()
    plt.show()
    return boxes, labels, scores

# Example usage:
# image_path = "/path/to/test_image.jpg"
# inference_image(model, image_path, device, conf_threshold=0.5)

In [ ]:
image_path = "/path/to/test_image.jpg"
inference_image(model, image_path, device, conf_threshold=0.5)

In [ ]:
def inference_video(model, video_path, output_path, device, conf_threshold=0.5):
    """Run inference on a video file and save annotated output with bounding boxes."""
    model.eval()
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Cannot open video {video_path}")
        return

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    CLASS_COLORS_BGR = {1: (100, 200, 255), 2: (0, 255, 0), 3: (0, 0, 255), 4: (255, 100, 0)}

    frame_count = 0
    print(f"Processing video: {total_frames} frames at {fps} FPS")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image_tensor = F.to_tensor(rgb_frame).to(device)

        with torch.no_grad():
            outputs = model([image_tensor])[0]

        keep = outputs['scores'] > conf_threshold
        boxes = outputs['boxes'][keep].cpu().numpy()
        labels = outputs['labels'][keep].cpu().numpy()
        scores = outputs['scores'][keep].cpu().numpy()

        for box, label, score in zip(boxes, labels, scores):
            x1, y1, x2, y2 = map(int, box)
            color = CLASS_COLORS_BGR.get(label, (200, 200, 200))
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cls_name = CLASS_NAMES[label - 1] if label <= len(CLASS_NAMES) else f"Class {label}"
            text = f'{cls_name}: {score:.2f}'
            (text_w, text_h), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(frame, (x1, y1 - text_h - 10), (x1 + text_w, y1), color, -1)
            cv2.putText(frame, text, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        cv2.putText(frame, f'Frame: {frame_count}/{total_frames}', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        out.write(frame)
        frame_count += 1
        if frame_count % 100 == 0:
            print(f"Processed {frame_count}/{total_frames} frames")

    cap.release()
    out.release()
    print(f"Video saved to: {output_path}")

# Example usage:
# input_video = "path/to/video.mp4"
# output_video = "path/to/output_video.mp4"
# inference_video(model, input_video, output_video, device, conf_threshold=0.5)